# Chương 2 — Phân tích học mô tả bằng Python

Notebook thực hành pipeline: **câu hỏi quản trị → grain → chất lượng dữ liệu →
KPI → OLAP → trực quan → insight hành động**.

**Tình huống:** phân tích hiệu quả chuỗi cửa hàng tiện lợi Ailogy Mart trong
Quý 2/2026 để hỗ trợ nhập hàng và phân bổ marketing.

> Dữ liệu mô phỏng phục vụ học tập, được tạo ngay trong notebook. Chọn
> **Runtime → Run all** trên Google Colab.

## Mục tiêu học tập

1. Xác định grain, dimension, measure và KPI.
2. Phát hiện/xử lý missing, duplicate, mã không nhất quán và outlier.
3. Thực hiện query, groupby, pivot, slice/dice, drill-down và đối soát.
4. Chọn biểu đồ đúng cho xu hướng, so sánh, quan hệ và phân phối.
5. Viết insight theo cấu trúc **Quan sát → Diễn giải → Hành động**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", palette="deep")
RANDOM_SEED = 2026

## 1. Sinh dữ liệu giao dịch thô

**Grain dự kiến:** một dòng = một dòng sản phẩm trong đơn hàng. Vì một đơn có
thể có nhiều sản phẩm nên `order_id` không duy nhất; khóa dòng là `line_id`.
Đoạn mã cố ý chèn lỗi chất lượng để luyện tập.

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = 900
dates = pd.to_datetime(rng.choice(pd.date_range("2026-04-01", "2026-06-30"), n))
store_map = {
    "Miền Bắc": ["Hà Nội", "Hải Phòng"],
    "Miền Trung": ["Đà Nẵng", "Huế"],
    "Miền Nam": ["TP.HCM", "Cần Thơ"]
}
regions = rng.choice(list(store_map), n, p=[0.32, 0.20, 0.48])
cities = [rng.choice(store_map[r]) for r in regions]
categories = rng.choice(["Đồ uống", "Thực phẩm", "Gia dụng", "Chăm sóc cá nhân", "Văn phòng phẩm"],
                        n, p=[.28, .31, .14, .17, .10])
unit_price = np.round(rng.lognormal(11.1, .45, n), -3).astype(int)
quantity = rng.integers(1, 7, n)
discount = rng.choice([0, .05, .10, .15], n, p=[.48, .25, .20, .07])
sales = np.round(unit_price * quantity * (1-discount), -3)
cost_ratio = rng.uniform(.55, .86, n)
profit = np.round(sales * (1-cost_ratio), -3)
marketing = np.round(rng.uniform(8_000, 70_000, n), -3)

raw = pd.DataFrame({
    "line_id": [f"L{i:05d}" for i in range(1, n+1)],
    "order_id": [f"DH{x:04d}" for x in rng.integers(1, 540, n)],
    "order_date": dates.astype(str), "region": regions, "city": cities,
    "category": categories, "quantity": quantity, "unit_price": unit_price,
    "discount": discount, "sales": sales.astype(int), "profit": profit.astype(int),
    "marketing_cost": marketing.astype(int)
})

# Tạo lỗi có chủ đích: thiếu, mã không nhất quán, outlier và 6 dòng trùng.
raw.loc[rng.choice(raw.index, 8, replace=False), "profit"] = np.nan
raw.loc[rng.choice(raw.index, 10, replace=False), "city"] = " tp.hcm "
raw.loc[rng.choice(raw.index, 5, replace=False), "category"] = None
raw.loc[25, ["sales", "profit"]] = [12_000_000, 4_500_000]
raw = pd.concat([raw, raw.sample(6, random_state=RANDOM_SEED)], ignore_index=True)
raw.head()

## Bài tập 1 — Hồ sơ dữ liệu và grain

Trước khi `groupby`, hãy trả lời:

- Một dòng đại diện cho điều gì? Khóa nào cần duy nhất?
- Cột nào là **dimension**, cột nào là **measure**?
- Có thể cộng `sales` theo dòng không? Có thể cộng `discount` không?

In [ ]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "n_unique": raw.nunique(dropna=True)
})
print(f"Kích thước dữ liệu: {raw.shape}")
display(profile)
print("Số line_id bị lặp:", raw["line_id"].duplicated().sum())

## Bài tập 2 — Làm sạch có kiểm soát

Quy tắc:

1. Parse ngày; chuẩn hoá chuỗi và mã thành phố.
2. Loại bản ghi trùng theo khóa grain.
3. Không tự tiện thay số liệu thiếu: đánh dấu và loại khỏi KPI cần trường đó.
4. Gắn cờ outlier bằng IQR; **không mặc định xoá** vì có thể là giao dịch thật.

In [ ]:
df = raw.copy()
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
for col in ["region", "city", "category"]:
    df[col] = df[col].astype("string").str.strip()
df["city"] = df["city"].str.lower().replace({"tp.hcm": "TP.HCM"})
df["city"] = df["city"].replace({
    "hà nội": "Hà Nội", "hải phòng": "Hải Phòng", "đà nẵng": "Đà Nẵng",
    "huế": "Huế", "cần thơ": "Cần Thơ"
})

duplicate_count = int(df.duplicated("line_id").sum())
df = df.drop_duplicates("line_id").copy()
df["category_missing"] = df["category"].isna()
df["profit_missing"] = df["profit"].isna()
# Nhãn trình bày không thay đổi cột gốc, giúp biểu đồ minh bạch về dữ liệu thiếu.
df["category_display"] = df["category"].fillna("Chưa phân loại")

q1, q3 = df["sales"].quantile([.25, .75])
iqr = q3 - q1
df["sales_outlier"] = ~df["sales"].between(q1 - 1.5*iqr, q3 + 1.5*iqr)

print("Dòng trùng đã loại:", duplicate_count)
print("Profit thiếu:", df["profit_missing"].sum())
print("Category thiếu:", df["category_missing"].sum())
print("Sales được gắn cờ outlier:", df["sales_outlier"].sum())

In [ ]:
assert df["line_id"].is_unique
assert pd.api.types.is_datetime64_any_dtype(df["order_date"])
expected_cities = {city for cities in store_map.values() for city in cities}
assert set(df["city"].dropna()) <= expected_cities
print("Các kiểm tra grain, kiểu ngày và chuẩn hoá thành phố: ĐẠT")

**Thảo luận:** giao dịch 12 triệu là lỗi hay đơn hàng lớn thật? Cần đối chiếu
nguồn nào trước khi sửa/xoá? So sánh mean và median để thấy tác động outlier.

In [ ]:
pd.DataFrame({
    "measure": ["Mean", "Median"],
    "all_rows": [df["sales"].mean(), df["sales"].median()],
    "without_flagged_outliers": [df.loc[~df["sales_outlier"], "sales"].mean(),
                                  df.loc[~df["sales_outlier"], "sales"].median()]
}).round(0)

## Bài tập 3 — Metric, KPI và truy vấn quản trị

KPI được định nghĩa:

- **Doanh thu:** tổng `sales` ở grain dòng đơn hàng.
- **Lợi nhuận:** tổng `profit` trên các dòng có dữ liệu hợp lệ.
- **Biên lợi nhuận:** `sum(profit) / sum(sales)`, không lấy trung bình tỷ lệ từng dòng.
- **Giá trị đơn hàng trung bình:** tổng doanh thu / số `order_id` duy nhất.

In [ ]:
valid_profit = df.dropna(subset=["profit"])
kpis = {
    "rows": len(df),
    "orders": df["order_id"].nunique(),
    "total_sales": df["sales"].sum(),
    "total_profit": valid_profit["profit"].sum(),
    "profit_margin": valid_profit["profit"].sum() / valid_profit["sales"].sum(),
    "average_order_value": df["sales"].sum() / df["order_id"].nunique()
}
pd.Series(kpis, name="value")

In [ ]:
# Query: 5 ngành hàng có lợi nhuận cao nhất tại Miền Nam
query_result = (df.query("region == 'Miền Nam'")
                .groupby("category", as_index=False, dropna=False)
                .agg(sales=("sales", "sum"), profit=("profit", "sum"),
                     orders=("order_id", "nunique")))
query_result["profit_margin"] = query_result["profit"] / query_result["sales"]
query_result.sort_values("profit", ascending=False).head()

### Cạm bẫy double counting khi join khác grain

`monthly_store_cost` có grain **thành phố–tháng**, còn giao dịch có grain
**dòng đơn hàng**. Join rồi cộng chi phí sẽ lặp chi phí cho mỗi giao dịch.

In [ ]:
df["month"] = df["order_date"].dt.to_period("M").astype(str)
monthly_store_cost = (df[["city", "month"]].drop_duplicates()
                      .assign(store_cost=50_000_000))
wrong_join = df.merge(monthly_store_cost, on=["city", "month"], how="left")
wrong_total = wrong_join["store_cost"].sum()
correct_total = monthly_store_cost["store_cost"].sum()
print(f"Sai khi cộng sau join: {wrong_total:,.0f} đ")
print(f"Đúng ở grain thành phố–tháng: {correct_total:,.0f} đ")

## Bài tập 4 — OLAP bằng Pivot Table

- **Pivot:** đổi các chiều giữa hàng/cột.
- **Slice:** cố định một chiều, ví dụ tháng 6.
- **Dice:** chọn nhiều giá trị của nhiều chiều.
- **Drill-down:** vùng → thành phố → dòng giao dịch.

In [ ]:
cube = pd.pivot_table(df, values="sales", index="region", columns="month",
                      aggfunc="sum", margins=True, margins_name="Tổng")
cube

In [ ]:
# Slice tháng 6 và dice hai vùng/hai ngành hàng
june_slice = df[df["month"].eq("2026-06")].groupby("region")["sales"].sum()
dice = (df[df["region"].isin(["Miền Bắc", "Miền Nam"])
           & df["category"].isin(["Đồ uống", "Thực phẩm"])]
        .pivot_table(values="sales", index="region", columns="category", aggfunc="sum"))
print("SLICE THÁNG 6")
display(june_slice)
print("DICE")
display(dice)

In [ ]:
# Drill-down: chọn vùng có doanh thu thấp nhất rồi xem thành phố
region_sales = df.groupby("region")["sales"].sum().sort_values()
focus_region = region_sales.index[0]
city_detail = (df[df["region"].eq(focus_region)]
               .groupby("city", as_index=False)["sales"].sum()
               .sort_values("sales"))
print("Vùng cần drill-down:", focus_region)
city_detail

## Bài tập 5 — Chọn biểu đồ theo câu hỏi

1. Xu hướng theo thời gian → line chart.
2. So sánh ngành hàng → bar chart.
3. Quan hệ marketing–doanh thu → scatterplot (không suy ra nhân quả).
4. Phân phối và outlier → boxplot.

In [ ]:
monthly = df.groupby("month", as_index=False).agg(sales=("sales", "sum"), profit=("profit", "sum"))
category_profit = (df.groupby("category_display", as_index=False)["profit"].sum()
                   .sort_values("profit", ascending=False))

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.lineplot(data=monthly, x="month", y="sales", marker="o", ax=axes[0,0], color="#4472C4")
axes[0,0].set_title("Doanh thu theo tháng")
sns.barplot(data=category_profit, x="profit", y="category_display", ax=axes[0,1], color="#70AD47")
axes[0,1].set_title("Lợi nhuận theo ngành hàng")
sns.scatterplot(data=df, x="marketing_cost", y="sales", hue="region", alpha=.65, ax=axes[1,0])
axes[1,0].set_title("Marketing và doanh thu (mối liên hệ quan sát)")
sns.boxplot(data=df, x="region", y="sales", showfliers=True, ax=axes[1,1], color="#ED7D31")
axes[1,1].set_title("Phân phối doanh thu dòng đơn theo vùng")
for ax in axes.flat:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()

**Câu hỏi:** scatterplot cho thấy tương quan không đồng nghĩa marketing gây ra
doanh thu. Hãy nêu ít nhất hai biến gây nhiễu có thể có (ví dụ quy mô cửa hàng,
vị trí, mùa vụ).

## Bài tập 6 — Dashboard tĩnh và ngưỡng KPI

Đổi `TARGET_MARGIN` để mô phỏng **parameter**. Đổi `SELECTED_REGION` để mô
phỏng **filter**. Trên Tableau, parameter là biến đầu vào cho công thức; filter
chỉ giới hạn các bản ghi hiện có.

In [ ]:
SELECTED_REGION = "Tất cả"   # thử: Miền Bắc, Miền Trung, Miền Nam
TARGET_MARGIN = 0.20

view = df.copy() if SELECTED_REGION == "Tất cả" else df[df["region"].eq(SELECTED_REGION)]
dash = view.groupby("month", as_index=False).agg(sales=("sales", "sum"), profit=("profit", "sum"))
dash["margin"] = dash["profit"] / dash["sales"]
overall_margin = view["profit"].sum() / view.loc[view["profit"].notna(), "sales"].sum()
status = "ĐẠT" if overall_margin >= TARGET_MARGIN else "CẢNH BÁO"

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.lineplot(data=dash, x="month", y="sales", marker="o", ax=axes[0], color="#4472C4")
axes[0].set_title(f"Xu hướng doanh thu — {SELECTED_REGION}")
region_margin = view.groupby("city", as_index=False).agg(sales=("sales","sum"), profit=("profit","sum"))
region_margin["margin"] = region_margin["profit"] / region_margin["sales"]
sns.barplot(data=region_margin.sort_values("margin"), x="margin", y="city", ax=axes[1], color="#70AD47")
axes[1].axvline(TARGET_MARGIN, color="red", linestyle="--", label="Mục tiêu")
axes[1].legend(); axes[1].set_title("Biên lợi nhuận theo thành phố")
fig.suptitle(f"KPI biên lợi nhuận: {overall_margin:.1%} — {status}", fontsize=15)
plt.tight_layout(); plt.show()

## Bài tập 7 — Đối soát và xuất dữ liệu sạch cho Tableau

Đối soát số dòng, tổng doanh thu và tổng lợi nhuận. File CSV đầu ra giữ cờ dữ
liệu thiếu/outlier để dashboard minh bạch, thay vì âm thầm loại bỏ.

In [ ]:
audit = pd.DataFrame({
    "check": ["Số dòng sạch", "Tổng sales", "Tổng profit hợp lệ"],
    "python_value": [len(df), df["sales"].sum(), df["profit"].sum()]
})
assert audit.loc[0, "python_value"] == df["line_id"].nunique()
assert df.groupby("region")["sales"].sum().sum() == df["sales"].sum()
display(audit)

output_file = "ailogy_mart_q2_2026_clean.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
print("Đã tạo:", output_file)

Nếu chạy trên Colab, bỏ chú thích hai dòng dưới để tải CSV về máy:

In [ ]:
# from google.colab import files
# files.download("ailogy_mart_q2_2026_clean.csv")

## Bài tập tổng hợp — Insight hỗ trợ quyết định

Viết ít nhất 3 insight theo mẫu:

- **Quan sát:** con số/mẫu nào nổi bật? Nêu đúng kỳ, nhóm, đơn vị.
- **Diễn giải:** ý nghĩa kinh doanh là gì? Phân biệt dữ kiện với giả thuyết.
- **Hành động:** ai làm gì, ở đâu, khi nào; KPI nào dùng để theo dõi?

Sau đó đề xuất dashboard Tableau gồm tối thiểu 4 worksheet (line, bar, scatter,
table/map), hierarchy `Region → City`, một Filter Action và một Highlight
Action. Ghi rõ nguồn, grain, thời điểm cập nhật, owner và giới hạn dữ liệu.

### Rubric tự đánh giá (10 điểm)

| Tiêu chí | Điểm |
|---|---:|
| Đúng grain; xử lý missing/duplicate/kiểu dữ liệu có giải thích | 2 |
| KPI đúng công thức; không double counting; đối soát khớp | 2 |
| OLAP/query đúng câu hỏi | 2 |
| Biểu đồ đúng mục đích, tiêu đề/nhãn/đơn vị rõ | 2 |
| Insight phân biệt quan sát–giả thuyết và dẫn tới hành động | 2 |